# Murphy 1 · Chapter 4 · Practice

From **Neuron Voxel Lab** (https://neuron-voxel-lab.vercel.app/#M4). Each exercise has a question, a hint, and an answer that stays hidden until you click.

**How to use:** try the starter cell first (fill in the `...` TODOs), then compare with the hidden solution cell (double-click its title to see the code), then open the answer.

## 4.1 · The coin's MLE from scratch  
*Section §4.2 · derive*

You flip a coin n times and see h heads. Write the log-likelihood of θ and find the θ that maximises it.

<details><summary><b>Hint</b></summary>

log L = h log θ + (n − h) log(1 − θ). Set its derivative to zero.

</details>

✍️ *Write your answer here (double-click to edit).*

<details><summary><b>Worked answer</b></summary>

d/dθ log L = h/θ − (n − h)/(1 − θ) = 0 ⇒ h(1 − θ) = (n − h)θ ⇒ h = nθ ⇒ θ̂ = h/n. The second derivative, −h/θ² − (n−h)/(1−θ)², is negative, so it's a maximum. The 'obvious' answer, heads over flips, is exactly what maximum likelihood gives (Murphy §4.2.3).

</details>

## 4.2 · The MLE of variance is biased  
*Section §4.2 · code*

Draw 100,000 samples of size n = 5 from N(0, 4). For each, compute the MLE variance (divide by n) and the unbiased version (divide by n − 1). Average each over all samples and compare with the true 4.

<details><summary><b>Hint</b></summary>

x.var(axis=1, ddof=0) vs ddof=1.

</details>

In [ ]:
# Your turn: replace each ... with code
import numpy as np
rng = np.random.default_rng(5)
x = rng.normal(0, 2, size=(100000, 5))
mle = ...  # TODO: divide by n
unb = ...  # TODO: divide by n - 1
print(f"MLE average {mle:.3f}   unbiased average {unb:.3f}   theory for MLE {4*4/5:.3f}")

In [ ]:
# @title Solution (double-click to show the code)
import numpy as np
rng = np.random.default_rng(5)
x = rng.normal(0, 2, size=(100000, 5))
mle = x.var(axis=1, ddof=0).mean()  # divide by n
unb = x.var(axis=1, ddof=1).mean()  # divide by n - 1
print(f"MLE average {mle:.3f}   unbiased average {unb:.3f}   theory for MLE {4*4/5:.3f}")

<details><summary><b>Answer and explanation</b></summary>

The MLE averages ≈ 3.2 = 4 × (n−1)/n: it underestimates, because measuring spread around the sample mean (which sits in the middle of your own points) always looks tighter than around the true mean. Dividing by n − 1 corrects this and averages ≈ 4.

Expected output:
```
MLE average 3.196   unbiased average 3.995   theory for MLE 3.200
```

</details>

## 4.3 · Beta-binomial updating on a grid  
*Section §4.6 · code*

Start with a Beta(2, 2) prior on a coin's bias. Observe 7 heads in 10 flips. On a grid of 1,001 θ values, compute the posterior, its mean, its MAP, and a 95% credible interval. Check the mean against the formula (a + h)/(a + b + n).

<details><summary><b>Hint</b></summary>

posterior ∝ θ^(a−1+h) (1−θ)^(b−1+n−h). Normalise by dividing by the sum; use the cumulative sum for the interval.

</details>

In [ ]:
# Your turn: replace each ... with code
import numpy as np
a, b, h, n = 2, 2, 7, 10
th = np.linspace(0, 1, 1001)
post = ...  # TODO: prior x likelihood (unnormalised)
post /= ...  # TODO: normalise on the grid
mean = (th * post).sum()
cdf = np.cumsum(post)
lo, hi = th[np.searchsorted(cdf, 0.025)], th[np.searchsorted(cdf, 0.975)]
print(f"mean {mean:.3f} (formula {(a+h)/(a+b+n):.3f})  MAP {th[post.argmax()]:.3f}  95% CI [{lo:.2f}, {hi:.2f}]")

In [ ]:
# @title Solution (double-click to show the code)
import numpy as np
a, b, h, n = 2, 2, 7, 10
th = np.linspace(0, 1, 1001)
post = th ** (a - 1 + h) * (1 - th) ** (b - 1 + n - h)  # prior x likelihood (unnormalised)
post /= post.sum()  # normalise on the grid
mean = (th * post).sum()
cdf = np.cumsum(post)
lo, hi = th[np.searchsorted(cdf, 0.025)], th[np.searchsorted(cdf, 0.975)]
print(f"mean {mean:.3f} (formula {(a+h)/(a+b+n):.3f})  MAP {th[post.argmax()]:.3f}  95% CI [{lo:.2f}, {hi:.2f}]")

<details><summary><b>Answer and explanation</b></summary>

Posterior = Beta(9, 5). Mean = 9/14 ≈ 0.643, MAP = 8/12 ≈ 0.667, 95% interval ≈ [0.39, 0.86]. The MLE would be 0.7; the prior pulls slightly toward 0.5, as if you'd already seen 1 head and 1 tail.

Expected output:
```
mean 0.643 (formula 0.643)  MAP 0.667  95% CI [0.39, 0.86]
```

</details>

## 4.4 · Bootstrap a confidence interval  
*Section §4.7 · code*

You have 40 skewed measurements (exponential, scale 3). Use 5,000 bootstrap resamples to estimate the standard error of the median and a 95% percentile interval.

<details><summary><b>Hint</b></summary>

rng.choice(data, size=len(data), replace=True), then take the median, repeated.

</details>

In [ ]:
# Your turn: replace each ... with code
import numpy as np
rng = np.random.default_rng(6)
data = rng.exponential(3.0, 40)
meds = ...  # TODO: resample and recompute
print(f"sample median {np.median(data):.3f}")
print(f"bootstrap SE  {meds.std():.3f}")
print(f"95% interval  [{np.percentile(meds, 2.5):.3f}, {np.percentile(meds, 97.5):.3f}]   true median {3*np.log(2):.3f}")

In [ ]:
# @title Solution (double-click to show the code)
import numpy as np
rng = np.random.default_rng(6)
data = rng.exponential(3.0, 40)
meds = np.array([np.median(rng.choice(data, size=len(data), replace=True)) for _ in range(5000)])  # resample and recompute
print(f"sample median {np.median(data):.3f}")
print(f"bootstrap SE  {meds.std():.3f}")
print(f"95% interval  [{np.percentile(meds, 2.5):.3f}, {np.percentile(meds, 97.5):.3f}]   true median {3*np.log(2):.3f}")

<details><summary><b>Answer and explanation</b></summary>

The bootstrap treats your sample as a stand-in for the population and reruns the experiment by resampling. The spread of the 5,000 medians approximates how much the median would change if you collected new data. No formula for the median's standard error is needed.

Expected output:
```
sample median 1.616
bootstrap SE  0.344
95% interval  [1.023, 2.269]   true median 2.079
```

</details>

## 4.5 · Ridge tames a wild polynomial  
*Section §4.5 · code*

Fit a degree-12 polynomial to 15 noisy sine points with ridge regression w = (ΦᵀΦ + λI)⁻¹Φᵀy, for λ = 0, 1e−6, 1e−3, 1. Print the largest |weight| and the test error for each.

<details><summary><b>Hint</b></summary>

Build Φ with np.vander(x, 13, increasing=True). Scale x to [−1, 1] first for numerical sanity.

</details>

In [ ]:
# Your turn: replace each ... with code
import numpy as np
rng = np.random.default_rng(0)
f = lambda x: np.sin(2 * np.pi * x)
xtr = rng.random(15); ytr = f(xtr) + rng.normal(0, 0.2, 15)
xte = rng.random(200); yte = f(xte) + rng.normal(0, 0.2, 200)
P = lambda x: np.vander(2 * x - 1, 13, increasing=True)
for lam in [0, 1e-6, 1e-3, 1]:
    A = P(xtr).T @ P(xtr) + lam * np.eye(13)
    w = ...  # TODO: ridge closed form
    te = np.mean((P(xte) @ w - yte) ** 2)
    print(f"lambda {lam:<6}  max|w| {np.abs(w).max():12.1f}   test MSE {te:.4f}")

In [ ]:
# @title Solution (double-click to show the code)
import numpy as np
rng = np.random.default_rng(0)
f = lambda x: np.sin(2 * np.pi * x)
xtr = rng.random(15); ytr = f(xtr) + rng.normal(0, 0.2, 15)
xte = rng.random(200); yte = f(xte) + rng.normal(0, 0.2, 200)
P = lambda x: np.vander(2 * x - 1, 13, increasing=True)
for lam in [0, 1e-6, 1e-3, 1]:
    A = P(xtr).T @ P(xtr) + lam * np.eye(13)
    w = np.linalg.solve(A, P(xtr).T @ ytr)  # ridge closed form
    te = np.mean((P(xte) @ w - yte) ** 2)
    print(f"lambda {lam:<6}  max|w| {np.abs(w).max():12.1f}   test MSE {te:.4f}")

<details><summary><b>Answer and explanation</b></summary>

With λ = 0 the weights are huge (thousands or more) and test error is bad: the curve is a fragile balance of giant terms. A small λ shrinks the weights by orders of magnitude and test error improves. Too much λ (1) flattens the curve and error goes up again: bias. λ is a dial between variance and bias.

Expected output:
```
lambda 0       max|w|       5419.2   test MSE 36.6593
lambda 1e-06   max|w|         58.3   test MSE 0.3761
lambda 0.001   max|w|          3.5   test MSE 0.0767
lambda 1       max|w|          0.7   test MSE 0.2671
```

</details>